# Chess Training Dataset Analysis
This notebook is a complete catalog of the existing chess data and formats available for use. It further outlines the parsing and ETL functions necessary to define the input data for the training of the V7P3R Chess AI vX. The notebook is structured as follows:

1. **Data Catalog**: A comprehensive list of chess datasets, including their formats, sources, and key features.
2. **Data Parsing**: Functions to read and parse the various chess data formats (e.g., PGN, FEN, CSV, JSON) into a standardized format suitable for feature extraction.
3. **Feature Definitions**: A detailed outline of the features to be extracted from the datasets, including board states, move sequences, player ratings, and game outcomes.
4. **ETL Functions**: Functions to extract, transform, and load the parsed data into the feature dataset that can be used for training the chess AI model.
5. **Training Architecture**: An overview of the training architecture, specifying data input formats, batch processing, and integration with the model training pipeline.



## Raw Data Catalog

The following is a location, type, and schema reference for each data type being ingested.

### Data Sources

In [48]:
# Step 1: Identify data source paths and types
import os
from pathlib import Path
from collections import defaultdict

# Define the directories to scan
data_directories = [
    r"E:\Programming Stuff\Chess Engines\Chess Engine Playground\engine-metrics\raw_data\game_records",
    r"E:\Programming Stuff\Chess Engines\Chess Engine Playground\engine-metrics\raw_data\training_data"
]

# Initialize datasets for each file type
pgn_files = []
csv_files = []
json_files = []
jsonl_files = []
db_files = []

# File type mapping
file_type_map = {
    '.pgn': pgn_files,
    '.csv': csv_files,
    '.json': json_files,
    '.jsonl': jsonl_files,
    '.db': db_files
}

# Scan directories
for directory in data_directories:
    dir_path = Path(directory)
    if dir_path.exists():
        for file_path in dir_path.rglob('*'):
            if file_path.is_file():
                file_extension = file_path.suffix.lower()
                if file_extension in file_type_map:
                    file_record = {
                        'filename': file_path.name,
                        'filepath': str(file_path),
                        'filetype': file_extension,
                        'filesize': file_path.stat().st_size,
                    }
                    file_type_map[file_extension].append(file_record)

# Calculate total sizes by file type
total_sizes = {}
for file_type, files in file_type_map.items():
    total_gb = sum(f['filesize'] for f in files) / (1024**3)
    total_sizes[file_type] = total_gb

# Display results
print(f"Found {len(pgn_files)} PGN files ({total_sizes['.pgn']:.2f} GB)")
print(f"Found {len(csv_files)} CSV files ({total_sizes['.csv']:.2f} GB)")
print(f"Found {len(json_files)} JSON files ({total_sizes['.json']:.2f} GB)")
print(f"Found {len(jsonl_files)} JSONL files ({total_sizes['.jsonl']:.2f} GB)")
print(f"Found {len(db_files)} DB files ({total_sizes['.db']:.2f} GB)")

# Display sample records (max 2 per file type)
print("SAMPLE PGN FILES (max 2):")
for pgn in pgn_files[:2]:
    print(f"  {pgn['filename']} ({round(pgn['filesize']/1000000,2):,} MB)")

print("SAMPLE CSV FILES (max 2):")
for csv in csv_files[:2]:
    print(f"  {csv['filename']} ({round(csv['filesize']/1000000,2):,} MB)")

print("SAMPLE JSON FILES (max 2):")
for json_file in json_files[:2]:
    print(f"  {json_file['filename']} ({round(json_file['filesize']/1000000,2):,} MB)")

print("SAMPLE JSONL FILES (max 2):")
for jsonl in jsonl_files[:2]:
    print(f"  {jsonl['filename']} ({round(jsonl['filesize']/1000000,2):,} MB)")

print("SAMPLE DB FILES (max 2):")
for db in db_files[:2]:
    print(f"  {db['filename']} ({round(db['filesize']/1000000,2):,} MB)")


Found 657 PGN files (1.70 GB)
Found 1 CSV files (0.84 GB)
Found 4 JSON files (0.00 GB)
Found 1 JSONL files (95.90 GB)
Found 1 DB files (0.95 GB)
SAMPLE PGN FILES (max 2):
  Engine Battle 20250728.pgn (0.24 MB)
  SlowMate Tournament 20250720.pgn (1.03 MB)
SAMPLE CSV FILES (max 2):
  lichess_db_puzzle.csv (903.33 MB)
SAMPLE JSON FILES (max 2):
  tournament_analysis_20250808.json (0.25 MB)
  tournament_analysis_Engine_Battle_20250809.json (0.06 MB)
SAMPLE JSONL FILES (max 2):
  lichess_db_eval.jsonl (102,967.48 MB)
SAMPLE DB FILES (max 2):
  lichess_db_puzzle.db (1,017.87 MB)


### Source Record Counts
An inventory of the number of records in each dataset, which will help in understanding the scale of data available for training.

In [64]:
# Calculate total pgn games
pgn_game_counts = {}
total_pgn_games = 0
for pgn in pgn_files:
    try:
        with open(pgn['filepath'], 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
            games = content.split('[Event ')[1:]  # Split on the start of each game
            pgn_game_counts[pgn['filename']] = len(games)
            total_pgn_games += len(games)
    except Exception as e:
        print(f"Error reading {pgn['filename']}: {e}")
        pgn_game_counts[pgn['filename']] = 0

print(f"Total PGN Games: {total_pgn_games}")


Total PGN Games: 2492511


In [65]:
# Calculate total csv positions
csv_position_counts = {}
total_csv_positions = 0
for csv in csv_files:
    try:
        with open(csv['filepath'], 'r', encoding='utf-8', errors='ignore') as f:
            position_count = sum(1 for line in f) - 1  # Subtract header
            csv_position_counts[csv['filename']] = position_count
            total_csv_positions += position_count
    except Exception as e:
        print(f"Error reading {csv['filename']}: {e}")
        csv_position_counts[csv['filename']] = 0

print(f"Total CSV Positions: {total_csv_positions}")



Total CSV Positions: 4914603


In [72]:
# Calculate total jsonl evaluations
jsonl_evaluation_counts = {}
total_jsonl_evaluations = 0
if False: # too large to run in notebook
    for jsonl in jsonl_files:
        try:
            with open(jsonl['filepath'], 'r', encoding='utf-8', errors='ignore') as f:
                evaluation_count = sum(1 for line in f)
                jsonl_evaluation_counts[jsonl['filename']] = evaluation_count
                total_jsonl_evaluations += evaluation_count
        except Exception as e:
            print(f"Error reading {jsonl['filename']}: {e}")
            jsonl_evaluation_counts[jsonl['filename']] = 0
    print(f"Total JSONL Evaluations: {total_jsonl_evaluations}")
else:
    total_jsonl_evaluations = 388000000 # from previous runs, since counting is too slow in notebook
    print(f"Approximate JSONL Evaluations: {total_jsonl_evaluations}")


Approximate JSONL Evaluations: 388000000


### Structured Output
Once data sources are known, all data must be brought into the system in a conformed layer for future feature processing. The format will be as follows:

    Structure Definition (88 bytes):
        - FEN hash (8 bytes, uinit64) - 64-bit hash of the FEN string for quick lookup and deduplication
        - Evaluation (2 bytes, int16) - centipawn eval (-32000 to 32000, null=32767)
        - Depth (1 byte, uint8) - depth value (0 to 128, null=255)
        - Time (ms) (4 bytes) - move time in milliseconds (0 to MAX_TIME, null=4294967295)
        - Clock (s) (2 bytes) - remaining clock time in seconds (0 to 7200, null=65535)
        - WDL (1 byte: W/D/L) - perspective based outcome flag (1/0/-1 respectively, null=127)
        - Material (2 bytes) - total material value (0 to 206, null=65535)
        - Phase (1 byte) - non pawn count phase values (0 to 24, null=255)
        - Piece count (1 byte) - number of pieces on board (0 to 32, null=255)
        - Reserved (66 bytes for future features)
        - Fen String - **Note:** currently included for additional feature extraction (to be removed from final feature dataset)


## Parsing and ETL Functions

### Imports and Helpers

In [ ]:
# Imports for data processing
import re
import pandas as pd
import chess.pgn
import json

# Set null, min, and max values
NULL_FEN_HASH = 0
NULL_EVAL = 32767
MIN_EVAL = -32000
MAX_EVAL = 32000
NULL_DEPTH = 255
MIN_DEPTH = 0
MAX_DEPTH = 128
NULL_TIME = 4294967295
MIN_TIME = 0
MAX_TIME = 4294967294
NULL_CLOCK = 65535
MIN_CLOCK = 0
MAX_CLOCK = 7200
NULL_WDL = 127
MIN_WDL = -1
MAX_WDL = 1
NULL_MATERIAL = 65535
MIN_MATERIAL = 0
MAX_MATERIAL = 206
NULL_PHASE = 255
MIN_PHASE = 0
MAX_PHASE = 24
NULL_PIECE_COUNT = 255
MIN_PIECE_COUNT = 0
MAX_PIECE_COUNT = 32
NULL_FEN = "NULL"

# Piece Values
PAWN_VALUE = 100
KNIGHT_VALUE = 320
BISHOP_VALUE = 330
ROOK_VALUE = 500
QUEEN_VALUE = 900
KING_VALUE = 0

# ----------------
# Helper Functions
# ----------------
def extract_eval_and_depth(comment_text: str):
    """Extract eval (cp) and depth from comment text."""
    if not comment_text:
        return NULL_EVAL, NULL_DEPTH

    # [%eval 0.80,4]
    m = re.search(r"\[%eval\s+([^\],\s]+)(?:[,/](\d+))?\]", comment_text)
    if m:
        raw_eval = m.group(1).strip()
        depth = int(m.group(2)) if m.group(2) else 0

        if raw_eval.startswith("#"):
            mate_n = raw_eval[1:]
            if mate_n.startswith("-"):
                return MIN_EVAL, depth
            return MAX_EVAL, depth

        try:
            cp = int(round(float(raw_eval) * 100))
            cp = max(MIN_EVAL, min(MAX_EVAL, cp))
            return cp, depth
        except ValueError:
            pass
    
    # Eval: 0.14
    m = re.search(r"Eval:\s*([+-]?\d+(?:\.\d+)?)", comment_text)
    if m:
        try:
            cp = int(round(float(m.group(1)) * 100))
            cp = max(MIN_EVAL, min(MAX_EVAL, cp))
            return cp, NULL_DEPTH
        except ValueError:
            pass
    
    # (d2-d4 d7-d5 Ng1-f3) +0.70/3 1
    m = re.search(r"\)\s*([+-]?\d+(?:\.\d+)?)(?:[/,](\d+))?", comment_text)
    if m:
        try:
            cp = int(round(float(m.group(1)) * 100))
            cp = max(MIN_EVAL, min(MAX_EVAL, cp))
            depth = int(m.group(2)) if m.group(2) else 0
            return cp, depth
        except ValueError:
            pass
    return NULL_EVAL, NULL_DEPTH


def extract_clk_seconds(comment_text: str):
    """Extract [%clk ...] and return remaining clock time in seconds."""
    if not comment_text:
        return None

    m = re.search(r"\[%clk\s+([0-9:.]+)\]", comment_text)
    if not m:
        return None

    clk_str = m.group(1).strip()
    parts = clk_str.split(":")
    try:
        if len(parts) == 3:
            h = int(parts[0])
            mm = int(parts[1])
            ss = float(parts[2])
            total = h * 3600 + mm * 60 + ss
        elif len(parts) == 2:
            mm = int(parts[0])
            ss = float(parts[1])
            total = mm * 60 + ss
        else:
            total = float(parts[0])

        sec = int(round(total))
        return max(MIN_CLOCK, min(MAX_CLOCK, sec))
    except ValueError:
        return None


def parse_time_control(tc_str: str):
    """
    Parse PGN TimeControl header into (base_seconds, increment_seconds).
    Handles formats: '300+5', '300', '180+2', '-' (unknown), '1/40' (moves/time).
    Returns (base, increment) in seconds, or (None, None) if unparseable.
    """
    if not tc_str or tc_str in ("-", "?", ""):
        return None, None

    # Moves-based format: 40/9000 (ignore increment, use per-move estimate)
    moves_match = re.match(r"(\d+)/(\d+)", tc_str)
    if moves_match:
        moves = int(moves_match.group(1))
        total = int(moves_match.group(2))
        base_per_move = total / moves
        return base_per_move, 0

    # Standard format: base+increment or base
    tc_match = re.match(r"(\d+(?:\.\d+)?)(?:\+(\d+(?:\.\d+)?))?", tc_str)
    if tc_match:
        base = float(tc_match.group(1))
        increment = float(tc_match.group(2)) if tc_match.group(2) else 0.0
        return base, increment

    return None, None


def calculate_move_time(
    prev_clk_side: float | None,
    clk_remaining: float | None,
    increment: float,
    base_seconds: float | None,
    move_number: int,
) -> int:
    """
    Calculate time spent on a move in seconds.

    Uses clock difference when available, accounting for increment added
    after the move. Falls back to a time-control-based estimate when
    clock data is missing.

    Args:
        prev_clk_side: Clock reading before this move (seconds), or None.
        clk_remaining:  Clock reading after this move (seconds), or None.
        increment:      Per-move increment in seconds (0 if none).
        base_seconds:   Total base time for the game (seconds), or None.
        move_number:    Full-move number (used for fallback estimate).

    Returns:
        Estimated move time in miliseconds (clamped 0 to MAX_TIME).
    """
    # Primary: clock difference accounting for increment
    if prev_clk_side is not None and clk_remaining is not None:
        # time_used = prev_clock - curr_clock + increment (increment is added after move)
        time_used = prev_clk_side - clk_remaining + increment
        return max(MIN_TIME, min(MAX_TIME, int(round(time_used))))

    return NULL_TIME

def calculate_game_phase(fen: str) -> int:
    """
    Calculates the game phase from a FEN string based on non-pawn material.
    Returns an integer from 0 (Pure Endgame) to 24 (Pure Middlegame).
    """
    # 1. Define standard engine phase values for each piece type
    # (Kings and Pawns are inherently 0, so they are excluded)
    PHASE_VALUES = {
        'n': 1, 'b': 1, 'r': 2, 'q': 4,  # Black pieces
        'N': 1, 'B': 1, 'R': 2, 'Q': 4   # White pieces
    }
    
    # 2. Isolate the piece placement section of the FEN (the first component)
    piece_placement = fen.split()[0]
    
    # 3. Sum up the phase scores for all surviving pieces
    current_phase_score = 0
    for character in piece_placement:
        if character in PHASE_VALUES:
            current_phase_score += PHASE_VALUES[character]
            
    # 4. Safety catch: clip to maximum starting value (24) just in case a puzzle 
    # FEN has custom promotional pieces that exceed normal starting material.
    if current_phase_score > 24:
        current_phase_score = 24
        
    return current_phase_score




### PGN files may contain game results and move sequences.
*used to store chess games in a standard format, including metadata like event, site, date, players, and the moves played.*

```
# Example PGN file content:

[Event "AI vs. AI Game"]
[Site "Local Computer"]
[Date "2025.06.06"]
[Round "#"]
[White "AI: viper via minimax"]
[Black "AI: None via random"]
[Result "1-0"]

1. Nh3 { Eval: 0.14 } 1... h6 { Eval: 0.72 } 2. Nf4 { Eval: 1.37 } 2... b5
{ Eval: 2.02 } 3. Nc3 { Eval: 2.74 } 3... Ba6 { Eval: 2.74 } 4. Nd3
{ Eval: 2.74 } 4... Nc6 { Eval: 2.02 } 5. Nc5 { Eval: 2.02 } 5... Rc8
{ Eval: 2.02 } 6. Nxa6 { Eval: 3.82 } 6... Rb8 { Eval: 3.82 } 7. Nxb8
{ Eval: 7.97 } 7... d5 { Eval: 9.83 } 8. Nxc6 { Eval: 11.30 } 8... a6
{ Eval: 13.38 } 9. Nxd8 { Eval: 18.43 } 9... g6 { Eval: 20.51 } 10. Nxd5
{ Eval: 20.31 } 10... Rh7 { Eval: 24.74 } 11. Nc6 { Eval: 23.82 } 11... f6
{ Eval: 27.25 } 12. Nxc7+ { Eval: 25.48 } 12... Kd7 { Eval: 29.98 } 13. Nxb5
{ Eval: 26.35 } 13... Ke6 { Eval: 30.85 } 14. Nd8+ { Eval: 24.28 } 14... Kd5
{ Eval: 30.28 } 15. Nc7+ { Eval: 25.78 } 15... Ke4 { Eval: 30.13 } 16. d4
{ Eval: 25.27 } 16... h5 { Eval: 1000025.49 } 1-0


# Example PGN file content (alternative):

[Event "casual blitz game"]
[Site "https://lichess.org/i6DnQ5hN"]
[Date "2025.10.10"]
[Round "?"]
[White "Viktor_Beltsov"]
[Black "c0br4_bot"]
[Result "0-1"]
[GameId "i6DnQ5hN"]
[UTCDate "2025.10.10"]
[UTCTime "14:27:40"]
[WhiteElo "1291"]
[BlackElo "1392"]
[BlackTitle "BOT"]
[Variant "Standard"]
[TimeControl "120+5"]
[ECO "C49"]
[Opening "Four Knights Game: Spanish Variation, Double Spanish"]
[Termination "Normal"]

1. e4 { [%clk 0:02:00] } 1... e5 { [%clk 0:02:00] } 2. Nf3 { [%clk 0:02:03] }
2... Nc6 { [%clk 0:02:05] } 3. Bb5 { [%clk 0:02:04] } 3... Nf6
{ [%clk 0:02:09] } ( 3... Nf6 4. Bxc6 bxc6 5. O-O { [%eval 0.80,4] } ) 4. Nc3
{ [%clk 0:02:07] } 4... Bb4 { [%clk 0:02:14] } ( 4... Bb4 5. Bxc6 bxc6 6. O-O
{ [%eval 0.62,4] } ) 5. a3 { [%clk 0:02:10] } 5... Bxc3 { [%clk 0:02:19] } (
5... Bxc3 6. dxc3 O-O 7. O-O { [%eval -0.80,4] } ) 6. bxc3 { [%clk 0:02:14] }
6... Nxe4 { [%clk 0:02:23] } ( 6... Nxe4 7. Qe2 Nxc3 8. dxc3
{ [%eval -0.45,4] } ) 7. d3 { [%clk 0:02:14] } 7... Nxc3 { [%clk 0:02:28] } (
7... Nxc3 8. Bg5 f6 9. Bxf6 { [%eval -4.78,4] } ) 8. Bg5 { [%clk 0:01:56] }
8... f6 { [%clk 0:02:33] } ( 8... f6 9. Qd2 Nxb5 10. a4 { [%eval -5.58,4] } )
9. Bh4 { [%clk 0:01:51] } 9... Nxd1 { [%clk 0:02:37] } ( 9... Nxd1 10. Rxd1 Nd4
11. Nxd4 { [%eval -8.72,4] } ) 0-1

```



In [50]:
# Parse out PGN files into structured format (supports multiple clk/eval formats)

def parse_pgn_file_to_dataframe(pgn_filepath: str) -> pd.DataFrame:
    """
    Parses a PGN file and extracts structured data into a DataFrame.
    Handles various clock and evaluation formats found in comments and headers.
    """
    with open(pgn_filepath, "r", encoding="utf-8", errors="ignore") as pgn_file:
        game = chess.pgn.read_game(pgn_file)

    if game is None:
        return pd.DataFrame()

    tc_str = game.headers.get("TimeControl", "")
    base_seconds, increment = parse_time_control(tc_str)

    result = game.headers.get("Result", "*")
    records = []
    board = game.board()

    fallback_base = base_seconds if (base_seconds and base_seconds > 0) else 60.0
    prev_clk = {chess.WHITE: fallback_base, chess.BLACK: fallback_base}
    inc_val = increment if increment is not None else 0.0

    for node in game.mainline():
        move = node.move
        mover = board.turn  # True for White, False for Black
        
        # Advance the board state to the next position
        board.push(move)
        fen = board.fen()
        full_move_number = board.fullmove_number
        combined_comment = node.comment if node.comment else ""
        
        # 1. CLOCK & MOVE TIME CALCULATIONS
        clk_remaining = extract_clk_seconds(combined_comment)
        node_clk = node.clock()
        if node_clk is not None:
            clk_remaining = max(MIN_CLOCK, min(MAX_CLOCK, int(round(node_clk))))
            
        move_time = calculate_move_time(
            prev_clk_side=prev_clk[mover],
            clk_remaining=clk_remaining,
            increment=inc_val,
            base_seconds=base_seconds,
            move_number=full_move_number,
        )
        
        # Update clock tracker for the next ply
        if clk_remaining is not None:
            prev_clk[mover] = clk_remaining

        # 2. FIXED MATERIAL BALANCE (Relative to White vs Black)
        piece_map = board.piece_map()
        material = sum(
            {1: PAWN_VALUE, 2: KNIGHT_VALUE, 3: BISHOP_VALUE, 4: ROOK_VALUE, 5: QUEEN_VALUE, 6: KING_VALUE}.get(p.piece_type, 0)
            * (1 if p.color == chess.WHITE else -1)
            for p in piece_map.values()
        )
        piece_count = len(piece_map)

        # 3. FIXED WDL PERSPECTIVE (Relative to the CURRENT side to move in the generated FEN)
        # Note: board.turn now represents the player whose turn it currently is in the FEN
        current_turn = board.turn 

        if result == "1-0":
            wdl = 1 if current_turn == chess.WHITE else -1
        elif result == "0-1":
            wdl = 1 if current_turn == chess.BLACK else -1
        elif result == "1/2-1/2":
            wdl = 0
        else:
            wdl = NULL_WDL

        # 4. FIXED EVALUATION PERSPECTIVE (Relative to the CURRENT side to move in the generated FEN)
        node_eval = node.eval()
        if node_eval is not None:
            # Dynamically extract score from the perspective of the side to move in the FEN
            relative_score = node_eval.relative() if hasattr(node_eval, 'relative') else node_eval.pov(current_turn)
            parsed_cp = relative_score.score(mate_score=32767)
            
            if parsed_cp is not None:
                eval_cp = max(MIN_EVAL, min(MAX_EVAL, int(parsed_cp)))
            else:
                eval_cp = NULL_EVAL
            
            node_depth = node.eval_depth()
            depth = int(node_depth) if node_depth is not None else NULL_DEPTH
        else:
            # Fallback to comment parsing if node.eval() is empty
            eval_cp, depth = extract_eval_and_depth(combined_comment)

        records.append(
            {
                "fen_hash": hash(fen) & 0xFFFFFFFFFFFFFFFF,
                "evaluation": eval_cp,
                "depth": depth,
                "time": move_time,
                "clock": clk_remaining if clk_remaining is not None else NULL_CLOCK,
                "wdl": wdl,
                "material": material,
                "phase": calculate_game_phase(fen),
                "piece_count": piece_count,
                "fen": fen
            }
        )

    return pd.DataFrame(records)


# For testing, we'll just parse one PGN file to verify the logic works end-to-end
target_pgn_num = 450
sample_pgn_filepath = pgn_files[target_pgn_num].get("filepath")
pgn_df = parse_pgn_file_to_dataframe(sample_pgn_filepath)
print(f"Parsed {len(pgn_df)} positions from: {pgn_files[target_pgn_num]['filename']}")
print(f"\nSchema:\n{pgn_df.dtypes}")
print("\nSample records:")
pgn_df.head(100)

Parsed 63 positions from: c0br4_bot games vs. morphe157bot.pgn

Schema:
fen_hash       uint64
evaluation      int64
depth           int64
time            int64
clock           int64
wdl             int64
material        int64
phase           int64
piece_count     int64
fen            object
dtype: object

Sample records:


,fen_hash,evaluation,depth,time,clock,wdl,material,phase,piece_count,fen
0,7726961334700359579,32767,255,2,180,-1,0,24,32,rnbqkbnr/pppppppp/8/8/2P5/8/PP1PPPPP/RNBQKBNR ...
1,16531948969094544818,32767,255,2,180,1,0,24,32,rnbqkbnr/pp1ppppp/2p5/8/2P5/8/PP1PPPPP/RNBQKBN...
2,12884213084504851780,32767,255,0,182,-1,0,24,32,rnbqkbnr/pp1ppppp/2p5/8/2P5/2N5/PP1PPPPP/R1BQK...
3,4252962062827265764,32767,255,1,181,1,0,24,32,rnbqkb1r/pp1ppppp/2p4n/8/2P5/2N5/PP1PPPPP/R1BQ...
4,12893771428484396415,32767,255,0,184,-1,0,24,32,rnbqkb1r/pp1ppppp/2p4n/8/2PP4/2N5/PP2PPPP/R1BQ...
...,...,...,...,...,...,...,...,...,...,...
58,16815480667297465798,32767,255,4,79,-1,1150,10,15,5rk1/pp3pp1/2pPp3/2N4Q/7R/8/1P2BK2/8 b - - 0 30
59,9031988524670042043,32767,255,1,216,1,1150,10,15,5rk1/pp4p1/2pPpp2/2N4Q/7R/8/1P2BK2/8 w - - 0 31
60,15969040171005775600,32767,255,4,77,-1,1150,10,15,5rk1/pp4pQ/2pPpp2/2N5/7R/8/1P2BK2/8 b - - 1 31
61,9909229952026605914,32767,255,0,218,1,1150,10,15,5r2/pp3kpQ/2pPpp2/2N5/7R/8/1P2BK2/8 w - - 2 32


### JSONL files may contain evaluation records from engine analysis

```jsonl
jsonl_evaluation_filepath = "evaluations/evaluations.jsonl"  # Path to the JSONL file containing evaluation records

jsonl_evaluation_format = {         # jsonl_evaluation_format describes the expected JSONL structure
    "fen": "",                      # the position FEN only contains pieces, active color, castling rights, and en passant square.
    "evals": [                      # List of evaluations at different depths
        {                           # Each evaluation contains:
            "knodes": 0,            # number of kilo-nodes searched by the engine
            "depth": 0,             # depth reached by the engine
            "pvs": [                # list of principal variations
                {                   # Each PV contains:
                    "cp": 0,        # centipawn evaluation. Omitted if mate is certain.
                    "mate": None,   # mate evaluation. Omitted if mate is not certain.
                    "line": ""      # principal variation, in UCI format.
                }
            ]
        }
    ]
}

```

In [51]:
# JSON Parsing

def parse_jsonl_file_to_dataframe(jsonl_filepath: str) -> pd.DataFrame:
    """
    Parses a JSONL file containing chess position evaluations into a structured DataFrame.
    Expects each line to be a JSON object with at least 'fen' and 'evals' fields.
    """
    # Open the jsonl and read the first 10 records
    json_data = []
    with open(jsonl_filepath, "r", encoding="utf-8", errors="ignore") as jsonl_file:
        for _ in range(100):  # Read the first 100 lines for sampling
            line = jsonl_file.readline()
            if not line:
                break
            try:
                record = json.loads(line)
                json_data.append(record)
            except json.JSONDecodeError:
                continue

    # Convert JSONL evaluation example to structured format
    jsonl_records = []

    # Loop through each individual record dictionary inside your list
    for record in json_data:
        
        # Safely look up fields to prevent KeyErrors
        fen_str = record.get("fen", "")
        evals = record.get("evals", [])
        
        # Fallback default values if the 'evals' list is empty
        cp_val = NULL_EVAL
        depth_val = NULL_DEPTH
        
        # FIX: Check if 'evals' exists and is a list before trying to access it
        if evals and isinstance(evals, list):
            # 1. Grab the highest search entry (the first item in the evals list)
            best_eval_entry = evals[0]
            depth_val = best_eval_entry.get("depth", NULL_DEPTH)
            
            # 2. Navigate into the 'pvs' list to grab the top move's centipawn score
            pvs_list = best_eval_entry.get("pvs", [])
            if pvs_list and isinstance(pvs_list, list):
                cp_val = pvs_list[0].get("cp", NULL_EVAL)

        # Calculate piece count from fen string (count pieces by counting uppercase and lowercase letters)
        piece_count = sum(c.isalpha() for c in fen_str)

        # Calculate material count from fen string
        material = sum(
          {"p": PAWN_VALUE, "n": KNIGHT_VALUE, "b": BISHOP_VALUE, "r": ROOK_VALUE, "q": QUEEN_VALUE, "k": KING_VALUE}.get(c.lower(), 0)
            for c in fen_str
        )

        # Calculate WDL from the perspective of the current side to move in the FEN
        board = chess.Board(fen_str)
        if (cp_val > 200 and board.turn == chess.WHITE) or (cp_val < -200 and board.turn == chess.BLACK):
            wdl = 1
        elif (cp_val < -200 and board.turn == chess.WHITE) or (cp_val > 200 and board.turn == chess.BLACK):
            wdl = -1
        else:
            wdl = 0


        # Append the structured record row
        jsonl_records.append(
            {
                "fen_hash": hash(fen_str) & 0xFFFFFFFFFFFFFFFF,
                "evaluation": cp_val,
                "depth": depth_val,
                "time": NULL_TIME,                       # No move time data in JSONL example
                "clock": NULL_CLOCK,                               # No clock data in JSONL example
                "wdl": wdl,                                 # WDL calculated from result and perspective
                "material": material,                     # Calculate material from fen string
                "phase": calculate_game_phase(fen_str),  # Calculated phase based on piece count
                "piece_count": piece_count,               # Fast calculation from fen string
                "fen": fen_str
            }
        )

    # Convert to DataFrame
    return pd.DataFrame(jsonl_records)


# Example JSONL evaluation data (simulating a single line from a JSONL file)
json_example = {
  "fen": "2bq1rk1/pr3ppn/1p2p3/7P/2pP1B1P/2P5/PPQ2PB1/R3R1K1 w - -",
  "evals": [
    {
      "pvs": [
        {
          "cp": 311,
          "line": "g2e4 f7f5 e4b7 c8b7 f2f3 b7f3 e1e6 d8h4 c2h2 h4g4"
        }
      ],
      "knodes": 206765,
      "depth": 36
    },
    {
      "pvs": [
        {
          "cp": 292,
          "line": "g2e4 f7f5 e4b7 c8b7 f2f3 b7f3 e1e6 d8h4 c2h2 h4g4"
        },
        {
          "cp": 277,
          "line": "f4g3 f7f5 e1e5 d8f6 a1e1 b7f7 g2c6 f8d8 d4d5 e6d5"
        }
      ],
      "knodes": 92958,
      "depth": 34
    },
    {
      "pvs": [
        {
          "cp": 190,
          "line": "h5h6 d8h4 h6g7 f8d8 f4g3 h4g4 c2e4 g4e4 g2e4 g8g7"
        },
        {
          "cp": 186,
          "line": "g2e4 f7f5 e4b7 c8b7 f2f3 b7f3 e1e6 d8h4 c2h2 h4g4"
        },
        {
          "cp": 176,
          "line": "f4g3 f7f5 e1e5 f5f4 g2e4 h7f6 e4b7 c8b7 g3f4 f6g4"
        }
      ],
      "knodes": 162122,
      "depth": 31
    }
  ]
}

# For testing, we'll just target a few records to verify the logic works end-to-end
target_jsonl_num = 0 # Select the target pgn file for sampling
sample_jsonl_filepath = jsonl_files[target_jsonl_num].get("filepath")
jsonl_df = parse_jsonl_file_to_dataframe(sample_jsonl_filepath)
# Display results
print(f"Parsed {len(jsonl_df)} positions from JSONL example")
print(f"\nSchema:\n{jsonl_df.dtypes}")
print("\nSample records:")
jsonl_df.head(100)


Parsed 100 positions from JSONL example

Schema:
fen_hash       uint64
evaluation      int64
depth           int64
time            int64
clock           int64
wdl             int64
material        int64
phase           int64
piece_count     int64
fen            object
dtype: object

Sample records:


,fen_hash,evaluation,depth,time,clock,wdl,material,phase,piece_count,fen
0,16761920439339547287,69,46,4294967295,65535,0,2890,6,16,7r/1p3k2/p1bPR3/5p2/2B2P1p/8/PP4P1/3K4 b - -
1,3264089356925031737,0,58,4294967295,65535,0,1930,4,11,8/4r3/2R2pk1/6pp/3P4/6P1/5K1P/8 b - -
2,17890743490332481190,32767,95,4294967295,65535,1,740,2,6,6k1/6p1/8/4K3/4NN2/8/8/8 w - -
3,668306496055212773,24,36,4294967295,65535,0,7800,24,31,r1b2rk1/1p2bppp/p1nppn2/q7/2P1P3/N1N5/PP2BPPP/...
4,12679734256395590299,32767,245,4294967295,65535,1,1100,2,10,6k1/4Rppp/8/8/8/8/5PPP/6K1 w - -
...,...,...,...,...,...,...,...,...,...,...
95,14958976351159410302,36,20,4294967295,65535,0,6400,12,28,rn2k2r/pp3p1p/2p1bp2/8/2p1PP2/2N5/PP4PP/R3KB1R...
96,6045900044838881196,1074,17,4294967295,65535,-1,2790,6,15,5k2/6pp/2B2P2/PPR3r1/3P4/7P/6P1/1b4K1 b - -
97,4203835621842392277,-85,30,4294967295,65535,0,5960,18,25,6k1/3q1pp1/6n1/b2p2Pp/2pP2b1/p1P5/P1BQrPPB/1R3...
98,8212813207060592698,44,28,4294967295,65535,0,6300,12,27,r1b1k2r/pp1n1p1p/2p2p2/5P2/2B1P3/2N5/PP4PP/R3K...


### CSV files may contain puzzles solutions and move sequences

```
csv_puzzle_headers = "PuzzleId,FEN,Moves,Rating,RatingDeviation,Popularity,NbPlays,Themes,GameUrl,OpeningTags"
csv_puzzle_example1 = "00sHx,q3k1nr/1pp1nQpp/3p4/1P2p3/4P3/B1PP1b2/B5PP/5K2 b k - 0 17,e8d7 a2e6 d7d8 f7f8,1760,80,83,72,mate mateIn2 middlegame short,https://lichess.org/yyznGmXs/black#34,Italian_Game Italian_Game_Classical_Variation"
csv_puzzle_example2 = "00sJ9,r3r1k1/p4ppp/2p2n2/1p6/3P1qb1/2NQR3/PPB2PP1/R1B3K1 w - - 5 18,e3g3 e8e1 g1h2 e1c1 a1c1 f4h6 h2g1 h6c1,2671,105,87,325,advantage attraction fork middlegame sacrifice veryLong,https://lichess.org/gyFeQsOE#35,French_Defense French_Defense_Exchange_Variation"
```

In [52]:
# CSV Parsing

def parse_csv_file_to_dataframe(csv_filepath: str) -> pd.DataFrame:
    """Parses a CSV file containing chess puzzles into a structured DataFrame."""
    # --- Load the file straight into raw_df, sampling only the first 10 rows ---
    raw_df = pd.read_csv(csv_filepath, nrows=10)

    # Convert csv puzzles to structured format
    csv_records = []

    # Process rows row-by-row (This now works because raw_df is defined above)
    for _, row in raw_df.iterrows():
        # 1. Initialize the board with this puzzle's starting FEN
        start_fen = str(row["FEN"]).strip()
        board = chess.Board(start_fen)
        
        # 2. Extract and split ALL available moves into an accessible list
        moves_list = str(row["Moves"]).strip().split()

        for p in range(4):  # Loop through plies 0 to 3 (0=starting position, 1=blunder, 2=solution, 3=post-solution)
            wdl = NULL_WDL  # Default WDL since we don't know the starting eval

            # 3. Step forward sequentially through plies
            if p > 0:
                # Ensure the puzzle sequence actually contains enough moves
                if len(moves_list) >= p:
                    # p=1 reads moves_list[0] (Blunder)
                    # p=2 reads moves_list[1] (Solution)
                    next_move = moves_list[p - 1] 
                    
                    try:                       
                        if p % 2 == 1:
                            wdl = 1
                        elif p % 2 == 0:
                            wdl = -1
                        move = board.parse_san(next_move)
                        board.push(move)  # State mutates normally to the next side's turn
                    except ValueError:
                        # If parse_san fails, it means we grabbed an illegal/wrong move
                        pass

            # 4. Gather the newly generated board parameters
            fen_str = board.fen()
            piece_map = board.piece_map()

            # 5. Calculate material score from piece map
            material_score = sum(
                {1: 100, 2: 320, 3: 330, 4: 500, 5: 900, 6: 0}.get(p.piece_type, 0) 
                for p in piece_map.values()
            )

            # Append mapped structured row entries
            csv_records.append(
                {
                    "fen_hash": hash(fen_str) & 0xFFFFFFFFFFFFFFFF,
                    "evaluation": NULL_EVAL,                    # Puzzles don't have evals
                    "depth": NULL_DEPTH,                           # Static puzzle starting points carry no search depth
                    "time": NULL_TIME,                     # No move timer fields present
                    "clock": NULL_CLOCK,                         # No running clocks active
                    "wdl": wdl,                             # No W/D/L data in CSV puzzle example
                    "material": material_score,
                    "phase": calculate_game_phase(fen_str),
                    "piece_count": len(piece_map),
                    "fen": fen_str
                }
            )

    # Wrap into the output DataFrame
    return pd.DataFrame(csv_records)



# For testing, we'll just target a few records to verify the logic works end-to-end
target_csv_num = 0 # Select the target pgn file for sampling
sample_csv_filepath = csv_files[target_csv_num].get("filepath")
puzzle_df = parse_csv_file_to_dataframe(sample_csv_filepath)

# Display results cleanly using Notebook table interface rules
print(f"Parsed {len(puzzle_df)} puzzle rows from CSV source.")
print(f"\nSchema:\n{puzzle_df.dtypes}")
print("\nSample records:")
puzzle_df.head(100)


Parsed 40 puzzle rows from CSV source.

Schema:
fen_hash       uint64
evaluation      int64
depth           int64
time            int64
clock           int64
wdl             int64
material        int64
phase           int64
piece_count     int64
fen            object
dtype: object

Sample records:


,fen_hash,evaluation,depth,time,clock,wdl,material,phase,piece_count,fen
0,12692498148359057762,32767,255,4294967295,65535,255,5450,18,20,r6k/pp2r2p/4Rp1Q/3p4/8/1N1P2R1/PqP2bPP/7K b - ...
1,8563531707081022544,32767,255,4294967295,65535,1,4950,16,19,r6k/pp2r2p/4Rp1Q/3p4/8/1N1P2b1/PqP3PP/7K w - -...
2,10580889253954726486,32767,255,4294967295,65535,-1,4450,14,18,r6k/pp2R2p/5p1Q/3p4/8/1N1P2b1/PqP3PP/7K b - - ...
3,17376777923838153003,32767,255,4294967295,65535,1,4450,14,18,r6k/pp2R2p/5p1Q/3p4/8/1N1P2b1/P1P3PP/1q5K w - ...
4,13877630389857923301,32767,255,4294967295,65535,255,4450,14,18,5rk1/1p3ppp/pq3b2/8/8/1P1Q1N2/P4PPP/3R2K1 w - ...
5,9526497575202576165,32767,255,4294967295,65535,1,4450,14,18,5rk1/1p3ppp/pq1Q1b2/8/8/1P3N2/P4PPP/3R2K1 b - ...
6,17133737013022108351,32767,255,4294967295,65535,-1,4450,14,18,3r2k1/1p3ppp/pq1Q1b2/8/8/1P3N2/P4PPP/3R2K1 w -...
7,4713594419462285369,32767,255,4294967295,65535,1,3950,12,17,3Q2k1/1p3ppp/pq3b2/8/8/1P3N2/P4PPP/3R2K1 b - -...
8,9634184740375417972,32767,255,4294967295,65535,255,1700,4,11,8/4R3/1p2P3/p4r2/P6p/1P3Pk1/4K3/8 w - - 1 64
9,13845948915817950000,32767,255,4294967295,65535,1,1700,4,11,8/5R2/1p2P3/p4r2/P6p/1P3Pk1/4K3/8 b - - 2 64


### Data Cleanup and Validation

In [54]:
# Combine all parsed DataFrames into a single master DataFrame for unified processing
combined_df = pd.concat([pgn_df, jsonl_df, puzzle_df], ignore_index=True)
print(f"Combined dataset contains {len(combined_df)} total positions.")

Combined dataset contains 203 total positions.
